# Injury Intelligence - EDA

Author: Angela Lekivetz

This notebook performs exploratory analysis of injury narratives to inform classification and RAG pipeline design. OSHA report is sourced from [OSHA Accident and Injury Data](https://www.kaggle.com/datasets/ruqaiyaship/osha-accident-and-injury-data-1517).

In [ ]:
import pandas as pd
import regex as re
import spacy

## 1. Load and Inspect Data

This dataset contains abstracts of the accidents and injuries of workers from 2015-2017. There is some structured data around the unstructured text abstracts, such as Degree of Injury, Body Part(s) Affected, and Construction End Use. 

In [ ]:
df = pd.read_csv('../data/osha_injuries_raw.csv')
print(df.info())
print(f'\nNulls: \n{df.isnull().sum()}')  
print(f'\nShape of dataframe: {df.shape}')
print(f'\nColumns: {df.columns}\n')

In [ ]:
# Keep only relevant columns
df = df[['Abstract Text', 'Event type', 'Event Keywords', 'Nature of Injury', 'Part of Body']]

## 2. Target Variable

The target variable is chosen to be event type, as it maps directly to the narrative text and has intuitive classes (fall, struck-by, etc.)

14 heavily imbalanced classes are found in the dataset, so a threshold of 140 was chosen to drop rare or ambiguous events. "Struck against" and "Fall (same level)" were removed as they had insufficient samples to train on and showed high confusion with similar classes ("Struck-by" and "Fall from elevation") in the initial model.

After filtering, 6 classes remained across 4,427 total rows.

In [ ]:
df_events = df['Event type'].value_counts() 
classes = df_events[df_events.values > 140].index.tolist()
df_filtered = df[df['Event type'].isin(classes)]
print(df_filtered['Event type'].value_counts())
print(len(df_filtered))

## 3. Text Analysis

Analyzing the abstract text column gives a mean of 72 words with a median of 59, meaning reasonably short narratives. 

Analyzing a few of the raw narratives, we see that the event type is often inferrable from the text, and that all narratives are written in third person and a consistent style. 

There is also a timestamp attached to the beginning of each narrative which will be stripped for this analysis. 

In [ ]:
print(df_filtered['Abstract Text'].isnull().sum())
df_filtered = df_filtered.copy()

df_filtered['text_len'] = df_filtered['Abstract Text'].str.split().str.len()
print(df_filtered['text_len'].describe())

In [ ]:
for event in df_filtered['Event type'].unique():
    sample = df_filtered[df_filtered['Event type'] == event]['Abstract Text'].iloc[0]
    print(f'[{event}]')
    print(sample[:200])
    print()

## 4. Preprocessing

Each narrative is cleaned by stripping the leading timestamp, lowercasing, removing numbers and punctuation, and filtering single character tokens.

For the TF-IDF baseline, lemmatization is also applied, reducing words to their base form (e.g. "operating" to "operate") and removing stop words. This reduces the feature space and helps the vectorizer treat different forms of the same word as one token.

DistilBERT receives the cleaned but non-lemmatized text, as it was pretrained on natural language and handles word forms internally.

In [ ]:
def clean_text(text):
    text = ','.join(text.split(',')[2:])
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join([w for w in text.split() if len(w) > 1])
    return text

df_filtered = df_filtered.copy()
df_filtered['clean_text'] = df_filtered['Abstract Text'].apply(clean_text)
print(df_filtered['clean_text'].head(10))

In [ ]:
nlp = spacy.load('en_core_web_sm')

def lemmatize(text):
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc if not token.is_stop])

sample = df_filtered['clean_text'].iloc[0]
print(df_filtered['Abstract Text'].iloc[0])
print('Before:', sample)
print('After:', lemmatize(sample))

In [ ]:
df_filtered = df_filtered.copy()
df_filtered['lemma_text'] = df_filtered['clean_text'].apply(lemmatize)

# Drop any nulls for classification step
df_filtered = df_filtered.dropna(subset=['lemma_text'])
df_filtered = df_filtered[df_filtered['lemma_text'] != '']

In [ ]:
print(df_filtered.isnull().sum())

## 5. Save Data

Save the filtered data as `osha_filtered.csv` in the `data/` folder for our next steps. 

In [ ]:
df_filtered.to_csv('../data/osha_filtered.csv', index=False)